# 12 — Ablation dữ liệu: visitor bất thường, nhóm lặp liên tiếp, ngưỡng phiên 60 phút

Notebook này **không đổi giao thức chính** (notebook 01–10 giữ nguyên artifact và kết quả). Mọi biến thể dữ liệu được dựng trong bộ nhớ, chỉ đánh giá trên **validation**; test chỉ được đọc lại từ danh sách Top-K đã lưu của lần chạy duy nhất ở notebook 10 (không chạy lại mô hình trên test).

| Mục | Câu hỏi | Cách làm |
| --- | --- | --- |
| A | Visitor hoạt động bất thường có làm lệch metric không? | Ngưỡng học **chỉ từ train**: phân vị 99,99% số sự kiện và số giao dịch theo (visitor, ngày UTC). Visitor vượt ngưỡng ở bất kỳ ngày nào của một tập bị gắn cờ trong tập đó. Báo cáo phụ: metric trên toàn bộ / bỏ visitor bị gắn cờ / chỉ visitor bị gắn cờ; độ nhạy: bỏ 10 visitor nhiều query nhất. |
| B | Nhóm lặp liên tiếp (reload, back) ảnh hưởng thế nào? | Ablation D1 trong protocol: bỏ nhóm có tập `(itemid, hành vi)` trùng nhóm liền trước trong cùng phiên, đánh lại `group_index`. Thiết kế 2×2: fit trên train gốc/đã gộp × đánh giá trên validation gốc/đã gộp. |
| C | Ngưỡng phiên 60 phút có tốt hơn 30 phút? | Chạy lại sessionize + chia tập (cùng mốc thời gian 0.8/0.9), dựng lại mẫu, vocabulary, `known_at`, fit lại mọi mô hình. Thiết kế 2×2: fit 30/60 phút × đánh giá 30/60 phút. |

- Mô hình: Popularity, S-POP, Repeat-last, ItemKNN, SKNN, V-SKNN, STAN, VSTAN, SR, GRU4Rec, SR-GNN với **đúng cấu hình đã chọn** trên validation (không tune lại). GRU4Rec/SR-GNN được huấn luyện lại với số epoch đã chọn khi dữ liệu fit thay đổi. Reranker không chạy ở đây vì nó được học trên chính validation (đánh giá trên validation sẽ là in-sample).
- Phép so sánh **cùng bộ query** (cột "đánh giá gốc") dùng paired bootstrap NDCG@20 theo protocol. Các cột "đánh giá trên dữ liệu biến đổi" có bộ query khác nên chỉ đọc như mô tả, không so trực tiếp.
- Đầu ra: `reports/data_ablation/metrics.json`, `reports/data_ablation/BAO_CAO_ABLATION_DU_LIEU.md`. Hàm dùng chung: `src/rr/ablation.py` (test: `tests/test_ablation.py`).

## 1. Cấu hình và kiểm tra đầu vào

In [ ]:
from pathlib import Path
import sys, time

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
from scipy.stats import kendalltau
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, require, utc_now, verify_inputs, write_json
from rr.ablation import (NEURAL, NON_NEURAL, build_samples, flag_heavy_visitors, heavy_visitor_thresholds,
                         known_at_for, merge_repeated_groups, score_models, sessionize, split_summary, topk_matrix)
from rr.evaluation import load_samples
from rr.metrics import evaluate, paired_bootstrap
from rr.samples import build_vocab

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
PROCESSED = ROOT / "data/processed"
OUT = ROOT / "reports/data_ablation"
SEED = PROTOCOL["seed"]
PAIRED = PROTOCOL["evaluation"]["paired_bootstrap"]
LOCKED_MS = PROTOCOL["locked_protocol"]["inactivity_threshold_ms"]
ALT_MS = 60 * 60 * 1000
DESCRIBE_MINUTES = [15, 30, 60, 120]
HEAVY_QUANTILE = 0.9999
TOP_VISITORS = 10
MODELS = NON_NEURAL + NEURAL
COLS = ["hit@20", "mrr@20", "ndcg@20"]
torch.set_num_threads(4)
require(LOCKED_MS == 1_800_000, "Ngưỡng phiên chính phải là 30 phút.")

prev = {n: read_json(ROOT / f"reports/{n}/metrics.json") for n in ["events_clean", "session_split", "metadata", "samples"]}
final_report = read_json(ROOT / "reports/final/metrics.json")
rehearsal = read_json(ROOT / "reports/final_rehearsal/metrics.json")
expected = {"data/processed/events_clean.parquet": prev["events_clean"]["output"],
            "data/processed/split_config.json": prev["session_split"]["split_config"]}
expected["data/processed/events_sessionized.parquet"] = prev["session_split"]["artifacts"]["data/processed/events_sessionized.parquet"]
for p in ["item_property_history", "item_catalog"]:
    expected[f"data/processed/{p}.parquet"] = prev["metadata"]["artifacts"][f"data/processed/{p}.parquet"]
expected.update(prev["samples"]["artifacts"])
expected.update(read_json(ROOT / "reports/final/frozen.json")["files"])  # protocol, validation reports, saved models
for r in [final_report, rehearsal]:
    expected[r["saved_topk"]["path"]] = r["saved_topk"]
require(final_report["protocol"]["sha256"] == fingerprint(PROTOCOL_PATH)["sha256"], "protocol.json khác lần chạy test.")
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
print("Đầu vào khớp SHA-256:", len(inputs), "tệp")
print(environment(), "| torch", torch.__version__)

## 2. Ví dụ biên trên dữ liệu nhỏ

- Ngưỡng phiên: khoảng nghỉ đúng 30 phút vẫn cùng phiên, 31 phút tách ở 30 phút nhưng không tách ở 60 phút.
- Gộp nhóm lặp: chỉ bỏ nhóm **liền sau** có cùng tập `(item, hành vi)` (thứ tự sự kiện trong nhóm không quan trọng); cùng item nhưng khác hành vi thì giữ.
- Visitor bất thường: đếm theo ngày, nên cùng số sự kiện rải trên hai ngày không bị gắn cờ.

In [ ]:
MIN = 60_000
def toy(rows):
    v, t, i, e = zip(*rows)
    return pd.DataFrame({"event_id": np.arange(len(rows)), "timestamp": np.array(t, dtype=np.int64),
                         "visitorid": np.array(v, dtype=np.int64), "itemid": np.array(i, dtype=np.int64),
                         "event": pd.Categorical(e, categories=["view", "addtocart", "transaction"])})

ex = toy([(1, 0, 10, "view"), (1, 30 * MIN, 11, "view"), (1, 61 * MIN, 12, "view")])
assert sessionize(ex, 30 * MIN, (10**9, 2 * 10**9)).sort_values("event_id")["session_id"].tolist() == [1, 1, 2]
assert sessionize(ex, 60 * MIN, (10**9, 2 * 10**9)).sort_values("event_id")["session_id"].tolist() == [1, 1, 1]

ex = toy([(1, 0, 1, "view"), (1, MIN, 1, "view"), (1, 2 * MIN, 2, "view"), (1, 3 * MIN, 2, "addtocart"),
          (1, 4 * MIN, 1, "view"), (1, 4 * MIN, 2, "view"), (1, 5 * MIN, 2, "view"), (1, 5 * MIN, 1, "view")])
m, st = merge_repeated_groups(sessionize(ex, 30 * MIN, (10**9, 2 * 10**9)))
assert m.sort_values("event_id")["timestamp"].drop_duplicates().tolist() == [0, 2 * MIN, 3 * MIN, 4 * MIN]
assert m.sort_values("event_id")["group_index"].drop_duplicates().tolist() == [1, 2, 3, 4]
assert st["groups_dropped"] == 2 and st["events_dropped"] == 3

thr = {"events_per_day": 1.5, "transactions_per_day": 0.0}
assert flag_heavy_visitors(toy([(8, 0, 5, "view"), (8, 1, 5, "view"), (9, 0, 5, "transaction")]), thr).tolist() == [8, 9]
assert flag_heavy_visitors(toy([(8, 0, 5, "view"), (8, 86_400_000, 5, "view")]), thr).tolist() == []
print("Đã kiểm tra ví dụ biên.")

## 3. Dữ liệu và cấu hình mô hình đã đóng băng

`sessionize()` với 30 phút phải tái tạo **đúng từng dòng** `session_id`, `group_id`, `group_index`, `split` của notebook 02, nếu không thì mọi so sánh phía sau vô nghĩa.

In [ ]:
clean = pd.read_parquet(PROCESSED / "events_clean.parquet", columns=["event_id", "timestamp", "visitorid", "event", "itemid"])
events = pd.read_parquet(PROCESSED / "events_sessionized.parquet",
                         columns=["event_id", "session_id", "visitorid", "group_id", "group_index", "itemid", "event", "timestamp", "split"])
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
split_cfg = read_json(PROCESSED / "split_config.json")
CUTOFFS = split_cfg["cutoff_ms"]
require(split_cfg["inactivity_threshold_ms"] == LOCKED_MS, "split_config khác ngưỡng đã chốt.")

key = ["session_id", "group_id", "group_index"]
redo = sessionize(clean, LOCKED_MS, CUTOFFS).sort_values("event_id").reset_index(drop=True)
orig = events.sort_values("event_id").reset_index(drop=True)
require(redo["event_id"].equals(orig["event_id"]) and all((redo[c].to_numpy() == orig[c].to_numpy()).all() for c in key)
        and (redo["split"].astype(str).to_numpy() == orig["split"].astype(str).to_numpy()).all(),
        "sessionize(30 phút) không tái tạo đúng notebook 02.")
del redo, orig
print("sessionize(30 phút) khớp notebook 02 trên mọi sự kiện.")

train_ev = events[events["split"] == "train"]
val_ev = events[events["split"] == "validation"]
train_q, train_t = load_samples(ROOT, "train")
val_q, val_t = load_samples(ROOT, "validation")
test_q, test_t = load_samples(ROOT, "test")
vocab_saved = pd.read_parquet(PROCESSED / "item_vocab.parquet")
require(build_vocab(train_ev).equals(vocab_saved[["itemid", "item_idx"]].reset_index(drop=True)), "Vocabulary khác notebook 04.")

ev_reports = {n: read_json(ROOT / f"reports/eval/{n}_validation.json") for n in ["popularity", "spop", "knn"]}
CFG = {"popularity": ev_reports["popularity"]["selected"],
       "spop_weights": ev_reports["spop"]["runs"][ev_reports["spop"]["best"]]["weights"],
       "knn_selected": ev_reports["knn"]["selected"],
       "gru": read_json(ROOT / "models/gru4rec/config.json"), "srgnn": read_json(ROOT / "models/srgnn/config.json"),
       "fallback_top_n": PROTOCOL["candidates"]["fallback_top_n"], "max_groups": PROTOCOL["knn"]["max_prefix_groups"],
       "max_len": PROTOCOL["long_sessions"]["gru_train_max_prefix_groups"] + 1, "seed": SEED, "width": 200, "root": ROOT}
print(f"train {len(train_q):,} query | validation {len(val_q):,} | test {len(test_q):,}")

RUNS, TIMINGS = {}, {}
def timed_run(name, *args, **kwargs):
    t0 = time.perf_counter()
    print(name)
    res, models = score_models(*args, **kwargs)
    TIMINGS[name] = round(time.perf_counter() - t0, 1)
    RUNS[name] = res
    print(f"  -> {TIMINGS[name] / 60:.1f} phút")
    return res, models

def table(res, part="all", cols=COLS):
    return pd.DataFrame({n: {**{c: r["report"][part][c] for c in cols}, "n_queries": r["report"][part]["n_queries"]}
                         for n, r in res.items()}).T

def sub(per_query, mask):
    return {**{c: float(per_query.loc[mask, c].mean()) for c in COLS}, "n_queries": int(mask.sum())}

def paired(a, b):
    return paired_bootstrap(a["per_query"]["ndcg@20"].to_numpy(), b["per_query"]["ndcg@20"].to_numpy(),
                            PAIRED["n_resamples"], PAIRED["seed"], PAIRED["level"])

## 4. Mốc so sánh: mô hình gốc trên validation gốc

Mô hình fit trên train gốc, GRU4Rec/SR-GNN nạp từ `models/`. NDCG@20 phải trùng tuyệt đối với `reports/final_rehearsal` (biến thể `train_only`), tức là đường code của notebook này tương đương notebook 10.

In [ ]:
ref, ref_models = timed_run("ref_train30_eval30", train_ev, val_ev, val_q, val_t, catalog, history, "known_at_train_ms", CFG, "load")
reh = rehearsal["results"]["train_only"]
gap = {n: abs(ref[n]["report"]["all"]["ndcg@20"] - reh[n]["main"]["all"]["ndcg@20"]) for n in MODELS}
require(max(gap.values()) < 1e-12, f"Mốc so sánh lệch diễn tập notebook 10: {gap}")
print("Khớp diễn tập notebook 10 cho", len(gap), "mô hình.")
display(table(ref).style.format({c: "{:.4f}" for c in COLS}))

## 5. Mục A — visitor hoạt động bất thường

Ngưỡng học từ train. Báo cáo cho validation (mọi mô hình) và cho test (chỉ VSTAN và GRU4Rec, là hai danh sách Top-K notebook 10 đã lưu; kết quả toàn tập được đối chiếu lại với `reports/final` trước khi tách nhóm).

In [ ]:
HEAVY = heavy_visitor_thresholds(train_ev, HEAVY_QUANTILE)
print("Ngưỡng từ train:", HEAVY)
samples = {"train": (train_q, train_t), "validation": (val_q, val_t), "test": (test_q, test_t)}
heavy_ids, heavy_stats, masks = {}, {}, {}
for sp, (q, t) in samples.items():
    ev = events[events["split"] == sp]
    heavy_ids[sp] = flag_heavy_visitors(ev, HEAVY)
    is_heavy = q["visitorid"].isin(heavy_ids[sp]).to_numpy()
    top = q["visitorid"].value_counts().head(TOP_VISITORS).index
    masks[sp] = {"heavy": is_heavy, "top": q["visitorid"].isin(top).to_numpy()}
    heavy_stats[sp] = {
        "flagged_visitors": int(len(heavy_ids[sp])),
        "event_share": float(ev["visitorid"].isin(heavy_ids[sp]).mean()),
        "query_share": float(is_heavy.mean()),
        "top_visitors_query_share": float(masks[sp]["top"].mean()),
        "max_queries_one_visitor": int(q["visitorid"].value_counts().iloc[0]),
        "all_repeat_share": {"heavy": float(q.loc[is_heavy, "all_repeat"].mean()), "other": float(q.loc[~is_heavy, "all_repeat"].mean())},
        "strong_target_share": {"heavy": float(q.loc[is_heavy, "target_has_strong"].mean()),
                                "other": float(q.loc[~is_heavy, "target_has_strong"].mean())},
    }
display(pd.DataFrame({sp: {k: v for k, v in s.items() if not isinstance(v, dict)} for sp, s in heavy_stats.items()}))
display(pd.DataFrame({sp: {f"{k}_{g}": v[g] for k in ["all_repeat_share", "strong_target_share"] for g, v in [(g, s[k]) for g in ["heavy", "other"]]}
                      for sp, s in heavy_stats.items()}))

def heavy_breakdown(per_query, m):
    return {"all": sub(per_query, np.ones(len(m["heavy"]), bool)), "without_flagged": sub(per_query, ~m["heavy"]),
            "flagged_only": sub(per_query, m["heavy"]), "without_top_visitors": sub(per_query, ~m["top"])}

A_val = {n: heavy_breakdown(r["per_query"], masks["validation"]) for n, r in ref.items()}
A_val_paired = {}
for n in MODELS:
    keep = ~masks["validation"]["heavy"]
    a = ref[n]["per_query"]["ndcg@20"].to_numpy()[keep]; b = ref["repeat_last"]["per_query"]["ndcg@20"].to_numpy()[keep]
    A_val_paired[n] = paired_bootstrap(a, b, PAIRED["n_resamples"], PAIRED["seed"], PAIRED["level"])
view = pd.DataFrame({n: {f"{part} ndcg@20": v[part]["ndcg@20"] for part in v} for n, v in A_val.items()}).T
view["Δ vs Repeat-last (bỏ cờ)"] = [A_val_paired[n]["mean_diff"] for n in view.index]
view["Δ thấp"] = [A_val_paired[n]["low"] for n in view.index]
display(view.style.format("{:.4f}"))
order_all = view["all ndcg@20"].rank(ascending=False); order_clean = view["without_flagged ndcg@20"].rank(ascending=False)
A_rank = {"kendall_tau_all_vs_without_flagged": float(kendalltau(order_all, order_clean).statistic),
          "order_all": view["all ndcg@20"].sort_values(ascending=False).index.tolist(),
          "order_without_flagged": view["without_flagged ndcg@20"].sort_values(ascending=False).index.tolist()}
print(A_rank)

saved = pd.read_parquet(PROCESSED / "final/test_train_only_topk.parquet")
A_test = {}
for n in ["vstan", "gru4rec"]:
    topk = topk_matrix(saved[saved["model"] == n], test_q["query_id"].to_numpy(), 20)
    rep, pq = evaluate(topk, test_q, test_t)
    require(abs(rep["all"]["ndcg@20"] - final_report["results"]["train_only"][n]["main"]["all"]["ndcg@20"]) < 1e-12,
            f"Top-K test đã lưu của {n} không khớp reports/final.")
    A_test[n] = heavy_breakdown(pq, masks["test"])
print("Test (train_only, Top-K đã lưu):")
display(pd.DataFrame({n: {p: v[p]["ndcg@20"] for p in v} for n, v in A_test.items()}).T.style.format("{:.4f}"))

## 6. Mục B — gộp nhóm lặp liên tiếp

Ô (fit, đánh giá): `gốc/gốc` là mốc ở mục 4. `gộp/gốc` trả lời "huấn luyện trên dữ liệu đã gộp có giúp bài toán gốc không" (cùng bộ query → paired bootstrap). `gốc/gộp` và `gộp/gộp` cho biết metric thay đổi thế nào khi các bước lặp hiển nhiên bị bỏ khỏi phép đo.

In [ ]:
merged, MERGE_STATS = merge_repeated_groups(events)
dropped_groups = np.setdiff1d(events["group_id"].unique(), merged["group_id"].unique())
val_q_dup = val_q["target_group_id"].isin(dropped_groups).to_numpy()   # target = exact repeat of the last prefix group
MERGE_STATS["validation_queries_with_duplicate_target"] = int(val_q_dup.sum())
mq_tr, mt_tr = build_samples(merged, "train")
mq_va, mt_va = build_samples(merged, "validation", first_query_id=len(mq_tr))
MERGE_STATS["queries"] = {"train": int(len(mq_tr)), "validation": int(len(mq_va))}
m_train = merged[merged["split"] == "train"]; m_val = merged[merged["split"] == "validation"]
require(build_vocab(m_train).equals(vocab_saved[["itemid", "item_idx"]].reset_index(drop=True)),
        "Gộp nhóm lặp không được làm đổi vocabulary (mỗi item bị bỏ đã xuất hiện ở nhóm liền trước).")
cat_m = known_at_for(catalog, m_train)
print(MERGE_STATS)

B1 = {n: {"duplicate_target": sub(r["per_query"], val_q_dup), "other_queries": sub(r["per_query"], ~val_q_dup)} for n, r in ref.items()}
display(pd.DataFrame({n: {p: v[p]["ndcg@20"] for p in v} for n, v in B1.items()}).T.style.format("{:.4f}"))

B_merged_merged, b_models = timed_run("train_merged_eval_merged", m_train, m_val, mq_va, mt_va, cat_m, history, "known_at_fit_ms",
                                      CFG, "train", fit_samples=(mq_tr, mt_tr))
B_merged_orig, _ = timed_run("train_merged_eval_orig", m_train, val_ev, val_q, val_t, cat_m, history, "known_at_fit_ms",
                             CFG, "reuse", neural_models=b_models)
B_orig_merged, _ = timed_run("train_orig_eval_merged", train_ev, m_val, mq_va, mt_va, catalog, history, "known_at_train_ms",
                             CFG, "reuse", neural_models=ref_models)
B_paired = {n: paired(B_merged_orig[n], ref[n]) for n in MODELS}
viewB = pd.DataFrame({n: {"gốc/gốc": ref[n]["report"]["all"]["ndcg@20"], "gộp/gốc": B_merged_orig[n]["report"]["all"]["ndcg@20"],
                          "Δ (gộp/gốc − gốc/gốc)": B_paired[n]["mean_diff"], "Δ thấp": B_paired[n]["low"], "Δ cao": B_paired[n]["high"],
                          "gốc/gộp": B_orig_merged[n]["report"]["all"]["ndcg@20"], "gộp/gộp": B_merged_merged[n]["report"]["all"]["ndcg@20"]}
                      for n in MODELS}).T
display(viewB.style.format("{:.4f}"))
del b_models

## 7. Mục C — ngưỡng phiên 60 phút

Mốc chia 0.8/0.9 tính trên timestamp sự kiện sạch nên **không phụ thuộc ngưỡng phiên**; chỉ việc gán phiên vào tập và `boundary_drop` thay đổi. Bảng mô tả cho 15/30/60/120 phút, rồi fit lại toàn bộ cho 60 phút.

In [ ]:
def cart_purchase_same_session(ev):
    tx = ev.loc[ev["event"].astype(str) == "transaction", ["visitorid", "itemid", "timestamp", "session_id"]]
    cart = ev.loc[ev["event"].astype(str) == "addtocart", ["visitorid", "itemid", "timestamp", "session_id"]]
    j = tx.merge(cart, on=["visitorid", "itemid"], suffixes=("", "_cart"))
    j = j[j["timestamp_cart"] <= j["timestamp"]].sort_values("timestamp_cart").groupby(["visitorid", "itemid", "timestamp"]).tail(1)
    return float((j["session_id"] == j["session_id_cart"]).mean())

DESCRIBE = {}
for minutes in DESCRIBE_MINUTES:
    s = sessionize(clean, minutes * MIN, CUTOFFS)
    summary = split_summary(s)
    n_groups = s.groupby("session_id")["group_index"].max()
    DESCRIBE[minutes] = {"sessions": int(len(n_groups)), "single_group_session_share": float((n_groups == 1).mean()),
                         "cart_then_purchase_same_session": cart_purchase_same_session(s),
                         **{f"{sp}_queries": summary[sp]["prefix_target_pairs"] for sp in ["train", "validation", "test"]},
                         "boundary_drop_sessions": summary["boundary_drop"]["sessions"],
                         "boundary_drop_events": summary["boundary_drop"]["events"]}
    if minutes * MIN == ALT_MS:
        s60 = s
    del s
display(pd.DataFrame(DESCRIBE))

tr60 = s60[s60["split"] == "train"]; va60 = s60[s60["split"] == "validation"]
q60_tr, t60_tr = build_samples(s60, "train")
q60_va, t60_va = build_samples(s60, "validation", first_query_id=len(q60_tr))
cat60 = known_at_for(catalog, tr60)
vocab60 = build_vocab(tr60)
SESSION60 = {"vocabulary_items": int(len(vocab60)), "vocabulary_items_30min": int(len(vocab_saved)),
             "queries": {"train": int(len(q60_tr)), "validation": int(len(q60_va))},
             "validation_target_oov_share": float((~t60_va["itemid"].isin(vocab60["itemid"])).mean())}
print(SESSION60)

C_60_60, c_models = timed_run("train60_eval60", tr60, va60, q60_va, t60_va, cat60, history, "known_at_fit_ms",
                              CFG, "train", fit_samples=(q60_tr, t60_tr))
C_60_30, _ = timed_run("train60_eval30", tr60, val_ev, val_q, val_t, cat60, history, "known_at_fit_ms", CFG, "reuse", neural_models=c_models)
C_30_60, _ = timed_run("train30_eval60", train_ev, va60, q60_va, t60_va, catalog, history, "known_at_train_ms", CFG, "reuse", neural_models=ref_models)
C_paired = {n: paired(C_60_30[n], ref[n]) for n in MODELS}
viewC = pd.DataFrame({n: {"30/30": ref[n]["report"]["all"]["ndcg@20"], "60/30": C_60_30[n]["report"]["all"]["ndcg@20"],
                          "Δ (60/30 − 30/30)": C_paired[n]["mean_diff"], "Δ thấp": C_paired[n]["low"], "Δ cao": C_paired[n]["high"],
                          "30/60": C_30_60[n]["report"]["all"]["ndcg@20"], "60/60": C_60_60[n]["report"]["all"]["ndcg@20"]}
                      for n in MODELS}).T
display(viewC.style.format("{:.4f}"))
del c_models

## 8. Lưu báo cáo

In [ ]:
def slim(res):
    return {n: {"report": r["report"], "fit_seconds": r["fit_seconds"], "query_seconds": r["query_seconds"]} for n, r in res.items()}

def md_table(frame, fmt="{:.4f}", first="Mô hình"):
    cols = list(frame.columns)
    head = f"| {first} | " + " | ".join(cols) + " |\n|" + " --- |" * (len(cols) + 1) + "\n"
    return head + "".join(f"| {i} | " + " | ".join(fmt.format(v) if isinstance(v, float) else str(v) for v in row) + " |\n"
                          for i, row in frame.iterrows())

require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
report = {
    "generated_at_utc": utc_now(), **environment(), "torch_version": torch.__version__,
    "notebook": "notebooks/12_ablation_du_lieu.ipynb",
    "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
    "inputs": inputs, "scope": "secondary analysis on validation; test only re-scored from saved train_only Top-K; main protocol unchanged",
    "configs": "frozen selections of reports/eval and models/ (no re-tuning); reranker excluded (trained on validation)",
    "reference_matches_rehearsal": True, "timings_seconds": TIMINGS,
    "A_heavy_visitors": {"rule": "visitor flagged in a split if on any UTC day its events or transactions exceed the train quantile",
                         "thresholds": HEAVY, "top_visitors_sensitivity": TOP_VISITORS, "stats": heavy_stats,
                         "flagged_visitorids": {sp: v.tolist() for sp, v in heavy_ids.items()},
                         "validation": A_val, "validation_paired_vs_repeat_last_without_flagged": A_val_paired,
                         "validation_model_order": A_rank, "test_saved_topk": A_test},
    "B_merge_repeated_groups": {"rule": PROTOCOL["repeat_targets"]["optional_ablation"], "stats": MERGE_STATS,
                                "validation_by_duplicate_target": B1, "paired_train_merged_vs_orig_on_orig_validation": B_paired},
    "C_session_threshold": {"locked_ms": LOCKED_MS, "alternative_ms": ALT_MS, "cutoff_ms": CUTOFFS,
                            "describe_by_minutes": {str(k): v for k, v in DESCRIBE.items()}, "session60": SESSION60,
                            "paired_train60_vs_train30_on_orig_validation": C_paired},
    "runs": {name: slim(res) for name, res in RUNS.items()},
    "status": "PASS",
}
write_json(OUT / "metrics.json", report)

pct = lambda x: f"{100 * x:.2f}%"
lines = [f"# Ablation dữ liệu (validation)\n\nSinh bởi `notebooks/12_ablation_du_lieu.ipynb` lúc {utc_now()}, protocol {PROTOCOL['version']}. "
         "Giao thức chính không đổi; đây là phân tích phụ. Mô hình dùng đúng cấu hình đã chọn, không tune lại; reranker không chạy "
         "(học trên validation). Số liệu chính: NDCG@20.\n",
         "\n## A. Visitor hoạt động bất thường\n",
         f"\nNgưỡng từ train (phân vị {HEAVY_QUANTILE}): > {HEAVY['events_per_day']:.1f} sự kiện/ngày hoặc > {HEAVY['transactions_per_day']:.0f} giao dịch/ngày.\n\n",
         "| Tập | Visitor bị gắn cờ | Tỷ lệ sự kiện | Tỷ lệ query | Top-10 visitor (tỷ lệ query) |\n| --- | --- | --- | --- | --- |\n",
         *[f"| {sp} | {s['flagged_visitors']} | {pct(s['event_share'])} | {pct(s['query_share'])} | {pct(s['top_visitors_query_share'])} |\n"
           for sp, s in heavy_stats.items()],
         "\n### Validation\n\n" + md_table(view),
         f"\nThứ tự mô hình (toàn bộ): {', '.join(A_rank['order_all'])}\n\nThứ tự mô hình (bỏ visitor bị gắn cờ): {', '.join(A_rank['order_without_flagged'])} "
         f"(Kendall tau = {A_rank['kendall_tau_all_vs_without_flagged']:.3f})\n",
         "\n### Test (train_only, Top-K đã lưu ở notebook 10)\n\n" +
         md_table(pd.DataFrame({n: {p: v[p]["ndcg@20"] for p in v} for n, v in A_test.items()}).T),
         "\n## B. Gộp nhóm lặp liên tiếp\n",
         f"\nBỏ {MERGE_STATS['groups_dropped']:,} / {MERGE_STATS['groups_before']:,} nhóm ({pct(MERGE_STATS['groups_dropped'] / MERGE_STATS['groups_before'])}); "
         f"{MERGE_STATS['validation_queries_with_duplicate_target']:,} / {len(val_q):,} query validation có target trùng hệt nhóm cuối prefix. "
         f"Query sau khi gộp: train {MERGE_STATS['queries']['train']:,}, validation {MERGE_STATS['queries']['validation']:,}.\n\n",
         "Ô = fit/đánh giá. Δ = paired bootstrap trên cùng bộ query validation gốc.\n\n" + md_table(viewB),
         "\n### Mô hình gốc, tách query validation theo target trùng nhóm cuối\n\n" +
         md_table(pd.DataFrame({n: {p: v[p]["ndcg@20"] for p in v} for n, v in B1.items()}).T),
         "\n## C. Ngưỡng phiên\n\n",
         md_table(pd.DataFrame({m: {c: f"{v:,}" if isinstance(v, int) else f"{v:.4f}" for c, v in d.items()} for m, d in DESCRIBE.items()}).T,
                  first="Ngưỡng (phút)"),
         f"\nPhiên 60 phút: vocabulary {SESSION60['vocabulary_items']:,} item (30 phút: {SESSION60['vocabulary_items_30min']:,}).\n\n",
         "Ô = ngưỡng fit/ngưỡng đánh giá (phút). Δ = paired bootstrap trên cùng bộ query validation 30 phút.\n\n" + md_table(viewC)]
(OUT / "BAO_CAO_ABLATION_DU_LIEU.md").write_text("".join(lines), encoding="utf-8", newline="\n")
print("Báo cáo:", rel(OUT / "metrics.json", ROOT), rel(OUT / "BAO_CAO_ABLATION_DU_LIEU.md", ROOT))

## 9. Kết luận

**A — Visitor bất thường.** Ngưỡng học từ train (> 145 sự kiện hoặc > 11 giao dịch trong một ngày UTC) gắn cờ 14 visitor ở validation (2,5% sự kiện nhưng 6,9% query) và 12 visitor ở test. Nhóm này khó hơn hẳn (NDCG@20 của GRU4Rec 0,261 so với 0,402 ở phần còn lại) và có tỷ lệ target thêm giỏ/mua gấp đôi (18,8% so với 8,4%), giống người mua sỉ hơn là bot xem trang. Bỏ họ ra làm mọi mô hình tăng 0,005–0,017 NDCG@20 nhưng **không đổi kết luận chính**: GRU4Rec vẫn đứng đầu, mọi mô hình kNN/neural vẫn hơn Repeat-last có ý nghĩa (cận dưới Δ > 0), test cho cùng xu hướng (VSTAN 0,334 → 0,345; GRU4Rec 0,351 → 0,362). Chỉ thứ tự trong nhóm kNN (chênh nhau < 0,003) thay đổi. → Giữ nguyên tập đánh giá chính, đưa bảng này vào báo cáo như phân tích độ vững.

**B — Nhóm lặp liên tiếp.** 22,9% query validation có target trùng hệt nhóm cuối prefix (reload/back). Trên phần này Repeat-last đạt 0,948 nên chúng đẩy metric lên mạnh; trên 77% query còn lại Repeat-last chỉ 0,183 và GRU4Rec 0,255 (hơn khoảng 39% tương đối, so với 9,6% trên toàn bộ). Gộp dữ liệu **huấn luyện** không có lợi: kNN gần như không đổi, còn GRU4Rec/SR-GNN/SR giảm 0,042–0,056 trên bài toán gốc và cũng không tốt hơn trên validation đã gộp (GRU4Rec 0,229 so với 0,255 khi fit trên dữ liệu gốc), vì mô hình mất tín hiệu "xem lại item vừa xem". → Không gộp dữ liệu huấn luyện; báo cáo thêm metric trên các query "không lặp hiển nhiên" để thấy rõ đóng góp thực của mô hình.

**C — Ngưỡng phiên 60 phút.** 60 phút thêm 3,9% query validation và tăng tỷ lệ "thêm giỏ → mua" nằm cùng phiên từ 89,9% lên 92,9%. Fit trên phiên 60 phút rồi đánh giá trên đúng bộ query 30 phút chỉ cải thiện rất nhỏ: kNN +0,001, SR +0,002, SR-GNN +0,005 (có ý nghĩa), GRU4Rec +0,0007 (khoảng tin cậy chứa 0), ItemKNN −0,003. Thứ tự mô hình không đổi. → Giữ 30 phút theo giao thức đã chốt; kết quả không nhạy với ngưỡng phiên trong khoảng 30–60 phút.

Giới hạn: mọi biến thể dùng cấu hình đã chọn cho dữ liệu gốc (không tune lại), nên biến thể có thể bị thiệt nhẹ; reranker không chạy vì được học trên validation.